# Notebook 6: Retrieval System

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the sixth notebook of the project.

In the previous notebooks, we completed:

1. Dataset collection and standardization
2. Text preprocessing
3. Document chunking
4. Embedding generation
5. FAISS vector indexing

In this notebook, we will build the retrieval system.

The retrieval system takes a user question, converts it into an embedding, searches the FAISS index, and returns the most relevant document chunks.

The goal of this notebook is to:

1. Load the FAISS index.
2. Load the chunk metadata.
3. Load the same embedding model used earlier.
4. Convert user queries into embeddings.
5. Retrieve top-k similar document chunks.
6. Display retrieved results clearly.
7. Save reusable retrieval results for the RAG notebook.

The output file from this notebook will be:

`sample_retrieval_results.csv`

This notebook prepares the retrieval component for:

`07_RAG_Pipeline.ipynb`

In [1]:
# Install required libraries
!pip install faiss-cpu sentence-transformers -q

# Import required libraries
import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# Find input file automatically
def find_input_file(file_name):
    """
    Searches for a file in common Kaggle locations.

    This helps because outputs from previous notebooks must usually
    be uploaded as input files in the next Kaggle notebook.
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    # Search inside Kaggle input folders
    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"{file_name} not found. Please upload it as input to this notebook."
    )


# Load FAISS index and metadata
def load_retrieval_files(index_file, metadata_file):
    """
    Loads the FAISS index and metadata file.
    """
    index_path = find_input_file(index_file)
    metadata_path = find_input_file(metadata_file)

    index = faiss.read_index(index_path)
    metadata = pd.read_csv(metadata_path)

    print("Retrieval files loaded successfully.")
    print("FAISS index file:", index_path)
    print("Metadata file:", metadata_path)
    print("Number of vectors in index:", index.ntotal)
    print("Metadata shape:", metadata.shape)

    return index, metadata


# Load embedding model
def load_embedding_model(model_name):
    """
    Loads the same sentence-transformer model used for chunk embeddings.
    """
    model = SentenceTransformer(model_name)

    print("Embedding model loaded successfully.")
    print("Model name:", model_name)

    return model


# Create query embedding
def embed_query(query, model):
    """
    Converts a user query into a normalized embedding vector.
    """
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return query_embedding.astype("float32")


# Retrieve top-k chunks
def retrieve_chunks(query, model, index, metadata, top_k=5):
    """
    Retrieves the top-k most relevant chunks for a user query.
    """
    query_embedding = embed_query(query, model)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        row = metadata.iloc[idx]

        results.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "chunk_id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "chunk_index": row["chunk_index"],
            "title": row["title"],
            "category": row["category"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(results)


# Display retrieved chunks
def display_retrieval_results(query, results_df):
    """
    Displays retrieval results in a readable format.
    """
    print("User Query:")
    print(query)

    print("\nTop Retrieved Chunks:\n")

    for _, row in results_df.iterrows():
        print("=" * 90)
        print("Rank:", row["rank"])
        print("Similarity Score:", round(row["score"], 4))
        print("Chunk ID:", row["chunk_id"])
        print("Document ID:", row["doc_id"])
        print("Title:", row["title"])
        print("Category:", row["category"])

        print("\nChunk Text Preview:\n")
        print(row["chunk_text"][:900])
        print()


# Run retrieval for multiple queries
def run_sample_queries(queries, model, index, metadata, top_k=5):
    """
    Runs retrieval on multiple sample queries and combines the results.
    """
    all_results = []

    for query in queries:
        results = retrieve_chunks(
            query=query,
            model=model,
            index=index,
            metadata=metadata,
            top_k=top_k
        )

        results["query"] = query
        all_results.append(results)

    return pd.concat(all_results, ignore_index=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.0 MB/s eta 0:00:00:00:0100:01


In [2]:
# Load FAISS index and metadata
index_file = "bbc_faiss_index.index"
metadata_file = "bbc_chunk_metadata.csv"

faiss_index, metadata = load_retrieval_files(index_file, metadata_file)


# Validate index and metadata
if faiss_index.ntotal != len(metadata):
    raise ValueError("FAISS index size does not match metadata rows.")

print("\nValidation successful.")
print("Each FAISS vector has a matching metadata row.")


# Preview metadata
print("\nMetadata Preview:")
display(metadata.head())


# Load embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = load_embedding_model(MODEL_NAME)


# Test retrieval with one query
sample_query = "What is happening in the economy and financial markets?"

retrieval_results = retrieve_chunks(
    query=sample_query,
    model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    top_k=5
)

display_retrieval_results(sample_query, retrieval_results)


# Display results as a table
print("\nRetrieval Results Table:")
display(retrieval_results[[
    "rank",
    "score",
    "chunk_id",
    "doc_id",
    "title",
    "category"
]])


# Test retrieval with multiple sample queries
sample_queries = [
    "What is happening in the economy and financial markets?",
    "What are the latest updates in sports?",
    "What news is related to politics and government?",
    "What is happening in technology companies?",
    "What entertainment news is being discussed?"
]

all_sample_results = run_sample_queries(
    queries=sample_queries,
    model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    top_k=3
)

print("\nMultiple Query Retrieval Results:")
display(all_sample_results[[
    "query",
    "rank",
    "score",
    "chunk_id",
    "title",
    "category"
]])


# Save sample retrieval results
output_file = "sample_retrieval_results.csv"

all_sample_results.to_csv(output_file, index=False)

print("\nSample retrieval results saved successfully.")
print("Output file:", output_file)


# Verify saved file
saved_results = pd.read_csv(output_file)

print("\nSaved file verified successfully.")
print("Saved file shape:", saved_results.shape)

display(saved_results.head())

Retrieval files loaded successfully.
FAISS index file: /kaggle/input/datasets/jahnavidulala/bbc-faiss-and-metadata/bbc_faiss_index.index
Metadata file: /kaggle/input/datasets/jahnavidulala/bbc-faiss-and-metadata/bbc_chunk_metadata.csv
Number of vectors in index: 8622
Metadata shape: (8622, 7)

Validation successful.
Each FAISS vector has a matching metadata row.

Metadata Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

,rank,score,chunk_id,doc_id,title,category
0,1,0.451837,1194_1,1194,IMF: UK set for slowest growth of G7 countries...,unknown
1,2,0.431972,8349_1,8349,UK borrowing costs at highest for a year after...,unknown
2,3,0.428470,6096_1,6096,US voters feel better about the economy. Will ...,unknown
3,4,0.426089,1717_1,1717,The Papers: Bank 'fails to calm market' as 'po...,unknown
4,5,0.421472,3179_1,3179,What do we know about the Silicon Valley and S...,unknown



Multiple Query Retrieval Results:


,query,rank,score,chunk_id,title,category
0,What is happening in the economy and financial...,1,0.451837,1194_1,IMF: UK set for slowest growth of G7 countries...,unknown
1,What is happening in the economy and financial...,2,0.431972,8349_1,UK borrowing costs at highest for a year after...,unknown
2,What is happening in the economy and financial...,3,0.428470,6096_1,US voters feel better about the economy. Will ...,unknown
3,What are the latest updates in sports?,1,0.498168,4739_1,"Premier League clubs to meet over TV deal, cal...",unknown
4,What are the latest updates in sports?,2,0.475855,7604_1,The Hollywood Olympics: All you need to know a...,unknown
5,What are the latest updates in sports?,3,0.452843,7392_1,Key rivalries to watch out for at Paris 2024,unknown
6,What news is related to politics and government?,1,0.440289,6724_1,The Papers: PM's pre-election speech and Niger...,unknown
7,What news is related to politics and government?,2,0.435814,7143_1,Has frantic election campaign actually grapple...,unknown
8,What news is related to politics and government?,3,0.408088,7331_1,From a buses bill to renter's rights - key poi...,unknown
9,What is happening in technology companies?,1,0.455085,5128_1,'We failed - why our dream eco-business collap...,unknown



Sample retrieval results saved successfully.
Output file: sample_retrieval_results.csv

Saved file verified successfully.
Saved file shape: (15, 9)


,rank,score,chunk_id,doc_id,chunk_index,title,category,chunk_text,query
0,1,0.451837,1194_1,1194,1,IMF: UK set for slowest growth of G7 countries...,unknown,The IMF cuts its UK forecast for 2023 and says...,What is happening in the economy and financial...
1,2,0.431972,8349_1,8349,1,UK borrowing costs at highest for a year after...,unknown,Analysts say the rise in borrowing costs is a ...,What is happening in the economy and financial...
2,3,0.428470,6096_1,6096,1,US voters feel better about the economy. Will ...,unknown,"Inflation is down and ""vibes"" are up - but you...",What is happening in the economy and financial...
3,1,0.498168,4739_1,4739,1,"Premier League clubs to meet over TV deal, cal...",unknown,Premier League clubs will be updated on new br...,What are the latest updates in sports?
4,2,0.475855,7604_1,7604,1,The Hollywood Olympics: All you need to know a...,unknown,"The new sporting events, the venues, the stars...",What are the latest updates in sports?
